**BfPortf 02 Custom target function**

This example shows how to use [bfportf](https://github.com/answering007/bfportf) with a metric of your own:
the search optimizes a custom function while the report keeps using the built in metrics.
[QuantStats](https://github.com/ranaroussi/quantstats) renders the visual report.

The examples need the `examples` extra (`uv sync --extra examples`).

In [ ]:
# Import section
import numpy as np
import pandas as pd
import quantstats as qs
from IPython.display import IFrame

import bfportf as bf

pd.set_option('display.precision', 3)

### Load data

`prices.parquet` is not tracked in git: the local copy from `assets/` is used when it exists, otherwise
the file is downloaded from the release asset.

In [ ]:
# Load data
from pathlib import Path
import urllib.request

import pandas as pd

# prices.parquet (4.5 MB, 500 symbols) is published as a GitHub release asset and is not tracked in git.
# A local copy is preferred: the repository layout (assets/prices.parquet), a file next to the notebook,
# or ../assets when the kernel runs inside the examples directory.
PRICES_CANDIDATES = (
    Path('assets/prices.parquet'),
    Path('prices.parquet'),
    Path('../assets/prices.parquet'),
)
PRICES_URL = 'https://github.com/answering007/bfportf/releases/latest/download/prices.parquet'
CSV_URL = 'https://raw.githubusercontent.com/answering007/bfportf/314397c/examples/prices.csv'

local_path = next((path for path in PRICES_CANDIDATES if path.exists()), None)

if local_path is None:
    # No local copy: take the release asset, or convert the original CSV while the asset is not published
    local_path = Path('assets/prices.parquet') if Path('assets').is_dir() else Path('prices.parquet')
    try:
        with urllib.request.urlopen(PRICES_URL, timeout=120) as response:
            local_path.write_bytes(response.read())
    except Exception:
        try:
            pd.read_csv(CSV_URL, index_col=0, parse_dates=['Date']).to_parquet(local_path)
        except Exception as error:
            raise RuntimeError(f'cannot get prices.parquet, download it manually: {PRICES_URL}') from error
        source = f'{CSV_URL} -> converted to {local_path}'
    else:
        source = f'{PRICES_URL} -> cached to {local_path}'
else:
    source = local_path

prices = pd.read_parquet(local_path)
if 'Date' in prices.columns:  # the file was written without an index
    prices = prices.set_index(pd.to_datetime(prices['Date'])).drop(columns=['Date'])

returns = bf.load_returns(prices)
bf.validate_returns(returns)
print(f'{returns.shape[0]} periods x {returns.shape[1]} symbols <- {source}')
returns.head()

### Split and scale

The split is a plain time series slice. The inverse volatility shares are estimated in-sample and the test
set is scaled with the same shares.

In [ ]:
split = int(len(returns) * 0.8)
df_train, df_test = returns.iloc[:split], returns.iloc[split:]

df_train_scaled, train_weights = bf.scale_return_data(df_train)
df_test_scaled = df_test * train_weights

print(f'Train shape: {df_train_scaled.shape}, test shape: {df_test_scaled.shape}')

### The custom metric

A metric is a callable that accepts a 1d array of returns and returns a number. The function below is the
annualized mean return divided by the dispersion raised to `1/exp`: a larger `exp` is a milder penalty for
volatility, so the parameter changes which portfolio wins.

In [ ]:
# Custom target function: the annualized mean return per unit of risk,
# the exp parameter damps the dispersion penalty (a larger exp is a milder penalty)
def mean_std_param_target(values: np.ndarray, exp: float = 3.0, periods: int = 252) -> float:
    """Annualized mean return divided by the dispersion raised to 1/exp."""
    dispersion = values.std()
    if exp == 0 or dispersion == 0:
        return np.nan
    return float(values.mean() * periods / dispersion ** (1 / exp))

A plain callable is accepted as well, but a `bf.Metric` also carries the name and the direction. The name
ends up in the result (`PortfolioResult.metric_name`), the direction is what the search reads instead of
guessing.

In [ ]:
# Wrap the function into a Metric: name, callable, direction
mean_std_3 = bf.Metric('mean_std_3', lambda values: mean_std_param_target(values, exp=3.0), maximize=True)

found_portfolios = bf.find_portfolios(
    returns=df_train_scaled,
    metric=mean_std_3,
    custom_weights=train_weights,
    max_number=3,
    verbose=True,
)

results = bf.results_to_frame(found_portfolios)
print(f'metric: {found_portfolios[0].metric_name} = {found_portfolios[0].value:.3f}')
results.head()

### Add the usual metrics to the results

The registry gives ready to use functions, the search direction is irrelevant here: every portfolio of the
results frame is evaluated with the same weights.

In [ ]:
for name in ('pnl', 'sharpe', 'sortino', 'max_drawdown'):
    results[name] = bf.calculate_metric(results, df_train_scaled, train_weights, bf.METRICS[name].func)

results

### The direction is a parameter

`maximize=False` flips the search: the same metric now finds the worst portfolio. `target_symbols` limits
the search to the given targets, a single symbol keeps the cell fast.

Undefined values are reported as `NaN` and such candidates are never selected, for example the profit
factor of a series without losing periods.

In [ ]:
worst = bf.find_portfolios(
    returns=df_train_scaled,
    metric=bf.Metric('mean_std_3', lambda values: mean_std_param_target(values, exp=3.0), maximize=False),
    custom_weights=train_weights,
    max_number=3,
    target_symbols=['SPY'],
    verbose=False,
)

worst[0].to_dict()

### QuantStats report

QuantStats builds a standalone HTML page, the notebook shows it in an iframe. The report is written next to
the notebook and is not tracked in git.

In [ ]:
# In-sample report
best = found_portfolios[0]

portfolio_series = bf.get_portfolio_return(
    returns=df_train_scaled,
    portfolio_symbols=best.symbols,
    return_weights=train_weights,
)
benchmark_series = bf.get_portfolio_return(
    returns=df_train_scaled,
    portfolio_symbols=['SPY'],
    return_weights=train_weights,
)

qs.reports.html(portfolio_series, benchmark=benchmark_series, output='qs_in_sample.html', title='In-sample')
IFrame(src='qs_in_sample.html', width='100%', height=900)

In [ ]:
# Out-of-sample report: the same symbols and the same weights on the untouched tail of the data
portfolio_series_out = bf.get_portfolio_return(
    returns=df_test_scaled,
    portfolio_symbols=best.symbols,
    return_weights=train_weights,
)
benchmark_series_out = bf.get_portfolio_return(
    returns=df_test_scaled,
    portfolio_symbols=['SPY'],
    return_weights=train_weights,
)

qs.reports.html(
    portfolio_series_out,
    benchmark=benchmark_series_out,
    output='qs_out_of_sample.html',
    title='Out-of-sample',
)
IFrame(src='qs_out_of_sample.html', width='100%', height=900)

The custom metric is only used by the search. Everything that describes the result (the report, the extra
metrics) is computed with the built in metrics, so different portfolios stay comparable.